In [1]:
import pyspark
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .master("local[*]") \
    .appName('test') \
    .getOrCreate()

In [2]:
!wget https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2025-11.parquet

--2026-03-07 15:20:31--  https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2025-11.parquet
Resolving d37ci6vzurychx.cloudfront.net (d37ci6vzurychx.cloudfront.net)... 2600:9000:2134:200:b:20a5:b140:21, 2600:9000:2134:3e00:b:20a5:b140:21, 2600:9000:2134:ae00:b:20a5:b140:21, ...
Connecting to d37ci6vzurychx.cloudfront.net (d37ci6vzurychx.cloudfront.net)|2600:9000:2134:200:b:20a5:b140:21|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 71134255 (68M) [binary/octet-stream]
Saving to: 'yellow_tripdata_2025-11.parquet'

     0K .......... .......... .......... .......... ..........  0% 10,0M 7s
    50K .......... .......... .......... .......... ..........  0%  159M 4s
   100K .......... .......... .......... .......... ..........  0%  433M 2s
   150K .......... .......... .......... .......... ..........  0%  653M 2s
   200K .......... .......... .......... .......... ..........  0%  676M 2s
   250K .......... .......... .......... .......... ..........

In [3]:
spark.version

'4.1.1'

In [5]:
df = spark.read.parquet('yellow_tripdata_2025-11.parquet')

In [6]:
df.printSchema()

root
 |-- VendorID: integer (nullable = true)
 |-- tpep_pickup_datetime: timestamp_ntz (nullable = true)
 |-- tpep_dropoff_datetime: timestamp_ntz (nullable = true)
 |-- passenger_count: long (nullable = true)
 |-- trip_distance: double (nullable = true)
 |-- RatecodeID: long (nullable = true)
 |-- store_and_fwd_flag: string (nullable = true)
 |-- PULocationID: integer (nullable = true)
 |-- DOLocationID: integer (nullable = true)
 |-- payment_type: long (nullable = true)
 |-- fare_amount: double (nullable = true)
 |-- extra: double (nullable = true)
 |-- mta_tax: double (nullable = true)
 |-- tip_amount: double (nullable = true)
 |-- tolls_amount: double (nullable = true)
 |-- improvement_surcharge: double (nullable = true)
 |-- total_amount: double (nullable = true)
 |-- congestion_surcharge: double (nullable = true)
 |-- Airport_fee: double (nullable = true)
 |-- cbd_congestion_fee: double (nullable = true)



In [7]:
df = df.repartition(4)

In [8]:
df.write.parquet('rep/')

In [30]:
df.registerTempTable('yellow')

In [31]:
df_result = spark.sql("""
SELECT 
    count(*)
FROM
    yellow
WHERE
    date_trunc('day',tpep_pickup_datetime)= date '2025-11-15'
""")

In [32]:
df_result.show()

+--------+
|count(1)|
+--------+
|  162604|
+--------+



In [39]:
df_result = spark.sql("""
SELECT 
    trip_distance,
    tpep_pickup_datetime,
    tpep_dropoff_datetime,
    (unix_timestamp( tpep_dropoff_datetime)-unix_timestamp( tpep_pickup_datetime))/3600 as hours
FROM
    yellow
order by 4 desc
""")

In [40]:
df_result.show()

+-------------+--------------------+---------------------+------------------+
|trip_distance|tpep_pickup_datetime|tpep_dropoff_datetime|             hours|
+-------------+--------------------+---------------------+------------------+
|       121.17| 2025-11-26 20:22:12|  2025-11-30 15:01:00| 90.64666666666666|
|         1.08| 2025-11-27 04:22:41|  2025-11-30 09:19:35| 76.94833333333334|
|          0.0| 2025-11-03 10:42:55|  2025-11-06 14:55:45| 76.21388888888889|
|         6.54| 2025-11-07 11:23:22|  2025-11-10 08:40:41| 69.28861111111111|
|         0.76| 2025-11-18 17:12:47|  2025-11-21 12:17:37| 67.08055555555555|
|         1.41| 2025-11-22 17:45:30|  2025-11-25 09:07:36| 63.36833333333333|
|          0.3| 2025-11-01 07:44:57|  2025-11-03 16:07:53|56.382222222222225|
|         5.41| 2025-11-27 14:34:56|  2025-11-29 14:43:48|48.147777777777776|
|         12.8| 2025-11-01 14:55:34|  2025-11-03 14:24:16| 47.47833333333333|
|          6.5| 2025-11-22 16:05:20|  2025-11-24 13:31:59| 45.44

In [41]:
!wget https://d37ci6vzurychx.cloudfront.net/misc/taxi_zone_lookup.csv

--2026-03-07 16:05:26--  https://d37ci6vzurychx.cloudfront.net/misc/taxi_zone_lookup.csv
Resolving d37ci6vzurychx.cloudfront.net (d37ci6vzurychx.cloudfront.net)... 2600:9000:2134:800:b:20a5:b140:21, 2600:9000:2134:e400:b:20a5:b140:21, 2600:9000:2134:5c00:b:20a5:b140:21, ...
Connecting to d37ci6vzurychx.cloudfront.net (d37ci6vzurychx.cloudfront.net)|2600:9000:2134:800:b:20a5:b140:21|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 12331 (12K) [text/csv]
Saving to: 'taxi_zone_lookup.csv'

     0K .......... ..                                         100% 2,83M=0,004s

2026-03-07 16:05:26 (2,83 MB/s) - 'taxi_zone_lookup.csv' saved [12331/12331]



In [42]:
df_zone = spark.read \
    .option("header", "true") \
    .csv('taxi_zone_lookup.csv')

+----------+-------------+--------------------+------------+
|LocationID|      Borough|                Zone|service_zone|
+----------+-------------+--------------------+------------+
|         1|          EWR|      Newark Airport|         EWR|
|         2|       Queens|         Jamaica Bay|   Boro Zone|
|         3|        Bronx|Allerton/Pelham G...|   Boro Zone|
|         4|    Manhattan|       Alphabet City| Yellow Zone|
|         5|Staten Island|       Arden Heights|   Boro Zone|
|         6|Staten Island|Arrochar/Fort Wad...|   Boro Zone|
|         7|       Queens|             Astoria|   Boro Zone|
|         8|       Queens|        Astoria Park|   Boro Zone|
|         9|       Queens|          Auburndale|   Boro Zone|
|        10|       Queens|        Baisley Park|   Boro Zone|
|        11|     Brooklyn|          Bath Beach|   Boro Zone|
|        12|    Manhattan|        Battery Park| Yellow Zone|
|        13|    Manhattan|   Battery Park City| Yellow Zone|
|        14|     Brookly

In [44]:
df_zone.registerTempTable('zone')

In [45]:
df_result = spark.sql("""
SELECT 
    Zone,
    count(PULocationID)
FROM
    yellow
left join zone on yellow.PULocationID= zone.LocationID
group by 1
order by 2 asc
""")

In [ ]:
df_result.show()